# Aspect-Based Sentiment Analysis

"Revenue rose, but costs climbed" holds two opinions. Aspect-based sentiment reads the polarity of each **aspect** a
sentence evaluates, relative to that aspect's **valence**: "up" is good news for revenue and bad news for costs. On the
synthetic finance corpus of Topics 17–18 the polarity is a deterministic function of the clause and the category, so
the Bayes accuracy is 1 and every question here is about *representation*. This notebook makes five points:
- no reader of the sentence alone gets both aspects of a contrast sentence, which caps it at $1 - m/2N = 0.911$
  (Prop 1);
- one attention layer whose query is the aspect is a **signed vote**: attention rescales each word's vote by a
  positive factor and never flips it (Thm 1(a));
- so when the aspect enters only through the query, "rose" and "fell" must vote the *same* way to read valence
  (Thm 1(b)), and no such layer reads negation at all (Thm 1(c)); bigram items fix it (Thm 1(d));
- attention weights are not identified by decisions: an equivalent model can put 0.998 of its attention on the aspect
  word without changing one decision (Thm 2);
- aggregated sentence by sentence, aspect polarities give back the document tone at Topic 17's Bayes ceiling (Prop 2).

Run from the repository root with
`uv run --with numpy==2.5.4 --with scipy==1.18.1 --with scikit-learn==1.9.1 --with jupyter --with ipykernel jupyter execute notebooks/aspect-based-sentiment/01_aspect_based_sentiment.ipynb`.

In [ ]:
import sys
from pathlib import Path

import numpy as np

here = Path.cwd()
sys.path.insert(0, str(here if (here / "aspect_based_sentiment.py").exists()
                       else here / "notebooks" / "aspect-based-sentiment"))
import aspect_based_sentiment as absa

fc, ls, ltc = absa.fc, absa.ls, absa.ltc
for split in ("train", "dev", "test"):
    print(split, "documents:", len(absa.docs(split)), "aspects:", len(absa.rows(split)))
y = absa.labels(absa.rows("test"))
print("test polarity counts (−, 0, +):", [int((y == k).sum()) for k in range(3)])
print("categories and valences:", dict(zip(absa.CATS, absa.VALENCE.tolist())))

## 1. An opinion has a target

Def 1: an aspect $a$ is a span of the sentence $x$ with a category $c(a)$ and a valence $v(c) \in \{-1, +1\}$; its
polarity $y \in \{-, 0, +\}$. A direction clause has $y = v(c)\,\delta$ for the move $\delta = \pm 1$; an evaluative
adjective carries $y$ itself; "not ADJ" reverses the adjective; a verb of expectation carries $v(c)$ times its sign.
Aspect spans and categories are gold here: finding them is span labeling (the extraction topics).

In [ ]:
rs = absa.rows("test")
i = absa.curated()[0]
print(rs[i].text)
for r in (rs[i], rs[i + 1]):
    print(f"  aspect {' '.join(r.toks[r.span[0]:r.span[1] + 1])!r}: category {absa.CATS[r.cat]}, "
          f"valence {absa.VALENCE[r.cat]:+d}, polarity {r.y - 1:+d}")
t = absa.accuracy_table()
print("construction counts:", t["n"])

## 2. The aspect-blind ceiling (Prop 1)

Every two-aspect sentence in the corpus has opposite polarities. Both aspects present a sentence-only reader with the
same input and different labels, so it reads at most one of them: accuracy $\le 1 - m/2N$. Topic 18's lexicon (with
its $k = 1$ negation scope) and an aspect-blind bigram logistic regression trained on the aspect labels both sit below
the bound, and the regression lands *exactly* on $1/2$ for contrast aspects.

In [ ]:
print("m =", sum(r.contrast for r in rs), " N =", len(rs), " bound =", round(absa.contrast_bound(rs), 6))
print("blind LR: lambda =", absa.blind_lr()["lam"], " dev =", round(absa.blind_lr()["dev"], 6))
for name in ("lexicon", "blind"):
    print(f"{name:8} all {t[name]['all']:.6f}  contrast {t[name]['contrast']:.6f}  "
          f"direction only {t[name]['by']['direction only']:.6f}  cost-like verb {t[name]['by']['verb, cost-like']:.6f}")

## 3. One layer, one vote (Def 2, Thm 1(a))

Def 2: items are tokens (and, optionally, adjacent pairs) with offsets $d_i$ from the aspect; scores are separable,
$s_i = \kappa(c, t_i) + \pi(c, d_i)$; values live in logit space, $z_i \in \mathbb R^3$; logits are
$\ell = \sum_i \alpha_i z_i + b_c$. Because $\sum_i \alpha_i = 1$,
$$\ell_k - \ell_l = \sum_i \alpha_i\, u_i^{kl}, \qquad u_i^{kl} = (z_i + b_c)_k - (z_i + b_c)_l,$$
and multiplying by $\sum_j e^{s_j} > 0$: $\operatorname{sign}(\ell_k - \ell_l) = \operatorname{sign}\sum_i e^{s_i} u_i^{kl}$.
For a fixed aspect, each decision is a linear threshold function of the bag of items, with weights that are positive
rescalings of fixed signed votes. The layer is Topic 9's attention with the aspect as a one-row query.

In [ ]:
absa.test_thm1a_vote_identity()
absa.test_layer_matches_topic9_attention()
absa.test_gradients_complex_step()
print("vote identity, Topic 9 cross-check, and complex-step gradients: ok")

## 4. Valence: the words that cannot mean what they say (Thm 1(b))

Take aspects $A$ (valence $+1$) and $B$ ($-1$) and the four sentences $\{A\ w_\uparrow, A\ w_\downarrow, B\ w_\uparrow,
B\ w_\downarrow\}$ labeled $+, -, -, +$. If the votes are shared across aspects (the aspect enters only through the
query), the vote signs are the same under $A$ and $B$, and the four strict inequalities force $u(w_\uparrow)$ and
$u(w_\downarrow)$ to be nonzero **with the same sign**: the query then amplifies $w_\uparrow$ for one aspect and
$w_\downarrow$ for the other, against a context that votes the other way. A linear program over every sign pattern
confirms it, and lets aspect-conditioned votes use the pattern $(+, -)$ as well.

In [ ]:
print("feasible (sign u_up, sign u_down), shared votes:", sorted(absa.valence_patterns(True)))
print("feasible, aspect-conditioned votes:            ", sorted(absa.valence_patterns(False)))
print("shared rung's ± votes:", {w: round(v, 3) for w, v in absa.shared_direction_votes().items()})
vq = absa.valence_quadruples()
for name in absa.RUNG_NAMES:
    print(f"{name:8} all-four-right on {len(vq)} valence quadruples: {absa.quadruple_scores(vq, name).mean():.6f}")

The trained shared rung found the theorem's configuration for one family only: *rose*, *fell*, *declined*, and
*decreased* all vote $-$, so it reads the three quadruples (rose, ·) and none of the six that pair *increased* or
*climbed* (which vote $+$) with a down word: $3/9 \approx 0.309$. Aspect-conditioned values (the aspect in the
values as well as the query, as ATAE-LSTM puts it in the input) read all of them.

## 5. Negation: what a vote cannot do (Thm 1(c)–(d))

For a fixed aspect, replacing the item type $t$ by $t'$ at offset $d$ changes $\sum_i e^{s_i}u_i^{\pm}$ by
$e^{\pi(c,d)}\big(\beta(t') - \beta(t)\big)$ with $\beta(t) = e^{\kappa(c,t)}u^{\pm}(c,t)$: a sign that no context and no
offset can change. Reading "T was ADJ₊" (+) and "T was ADJ₋" (−) needs $\beta(\text{ADJ}_-) < \beta(\text{ADJ}_+)$;
reading "T was not ADJ₊" (−) and "T was not ADJ₋" (+) needs the reverse. So no layer of Def 2 with token-level items
reads negation — Topic 17's XOR, now between two words of the context. A bigram item "not ADJ" escapes it.

In [ ]:
rng = absa._rng(3225)
ws = [(1.0, 1.0)] + [tuple(rng.uniform(0.01, 10, 2)) for _ in range(20)]
print("LP feasible with unigram items:", any(absa.negation_feasible(False, w) for w in ws),
      " with bigram items:", all(absa.negation_feasible(True, w) for w in ws))
nq = absa.negation_quadruples()
for name in absa.RUNG_NAMES:
    print(f"{name:8} all-four-right on {len(nq)} negation quadruples: {absa.quadruple_scores(nq, name).mean():.6f}")

Both unigram rungs read *no* negation quadruple, as the theorem demands; the bigram rung reads every one. So do
the two aspect-blind readers: Topic 18's "a negator within one token flips the sign" is a hand-written bigram feature,
and the blind regression has the bigrams "not strong", "not weak" among its features.
The whole ladder, by construction:

In [ ]:
cols = absa.CONSTRUCTIONS
print(f"{'':9}" + "".join(f"{c[:10]:>11}" for c in cols) + f"{'contrast':>11}{'all':>9}")
for name in absa.RUNG_NAMES:
    print(f"{name:9}" + "".join(f"{t[name]['by'][c]:11.3f}" for c in cols)
          + f"{t[name]['contrast']:11.3f}{t[name]['all']:9.3f}")

## 6. Attention is not the explanation (Thm 2)

Shift every score of type $t$ by $\log r_t$ and map its value $z \mapsto (z + b)/r_t - b$. Then every logit vector is
multiplied by $\lambda(x) = \sum_i e^{s_i} / \sum_i e^{s_i} r_{t_i} > 0$: every decision survives, every vote
$\alpha_i(z_i + b)$ is scaled by the same $\lambda(x)$, and the attention can be pushed onto any words we like. We push
it onto the aspect-term words of the trained bigram rung with $r = 10^6$.

In [ ]:
idf = absa.identifiability()
print("same decision on all", len(rs), "test aspects:", idf["same_argmax"])
print(f"attention on aspect-term words: before {idf['mass_before']:.6f}, after {idf['mass_after_mean']:.6f}"
      f" (min {idf['mass_after_min']:.6f})")
print(f"logit scale lambda: {idf['lam_min']:.6f} … {idf['lam_max']:.6f};  vote deviation {idf['vote_dev']:.1e}")
lab = absa.viz_payload()["identifiability"]
print("p(decision) on the lab's sentence as r grows:", [max(s["p"]) for s in lab["steps"]])

Decisions are identified; attention maps are not. Probabilities are not preserved either: $\lambda(x)$ acts as an
input-dependent temperature (Topic 16), and at $r = 10^6$ the confidence has flattened to $1/3$. What survives is the
vote profile $\alpha_i(z_i + b)$, the logit-space analogue of Kobayashi et al.'s $\lVert \alpha f(x)\rVert$.

## 7. From aspects to tone (Prop 2)

A document of tone $t$ has $k \in \{5, \dots, 8\}$ tone-bearing sentences whose polarities are i.i.d. from the tone's
mixture, and nothing else in the text depends on $t$; so the Bayes rule on the polarity counts is Topic 17's full-text
ceiling. A reader that misreads a fraction $\varepsilon$ of those sentences moves the accuracy by at most
$E[k]\,\varepsilon$ (a union bound).

In [ ]:
ag = absa.aggregation()
print("tone sentences:", ag["sentences"], " mean k:", round(ag["mean_k"], 6),
      " k = 5..8:", [ag["ks"].count(k) for k in range(5, 9)])
print(f"gold aggregation {ag['gold']:.6f}, bigram-rung aggregation {ag['pred']:.6f}, ceiling {ltc.bayes_ceiling():.6f}"
      f" (SE {ag['se']:.6f})")
print("reader errors:", ag["errors"], "in", ag["wrong_docs"], "documents")
for k, v in absa.document_baselines().items():
    print(f"  {k:28} {v:.6f}")

## 8. EDGAR: aspects in real MD&A

The excerpts carry no aspect gold, so these are readings, not scores. Out of the corpus's templates the reader has no
notion of "neutral": it calls descriptive mentions polar at about even odds, where the lexicon (no polar words in
sight) calls four of five neutral.

In [ ]:
for e in absa.edgar_readings():
    print(f"{e['company']:26} {e['term']!r:16} {e['pred']:9} p = {np.round(e['p'], 3)}  lexicon: {e['lexicon']}")

## The whole harness

The `.py` runs every check above (and the lab's data mirror) in one go.

In [ ]:
failed = absa._run_harness()
assert failed == 0

## Where this sits

Topic 18 left "direction only" and "cost-like verb" sentences to a reader that knows the aspect; this topic is that
reader, and its limits are the limits of one vote. Fine-tuning (Topic 20) trains a contextual encoder underneath, the
standard escape from Thm 1; the extraction topics (21–22) find the aspect spans this notebook took as gold; and the
capstone scores earnings calls aspect by aspect. formalML's `kernel-regression` develops the Nadaraya–Watson estimator
behind the position kernel; formalRAG's `cross-encoders-reranking` is the same interaction question for retrieval.